# Web Search on Amazon Bedrock AgentCore with a LangChain agent

[Web Search on Amazon Bedrock AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html)
is a managed tool that returns source attributed results from the public web. This
notebook builds an agent that uses it to answer questions about things that happened
after the model's training cutoff, and to cite where each fact came from.

`create_web_search_toolkit` returns an ordinary LangChain tool, so the agent code is
the same as it would be for any other tool. Searches are signed with SigV4 from your
ambient AWS credentials. There is no web search API key.

## What you need

- An AWS account with the web search connector enabled. It is enabled per account, and
  `CreateGatewayTarget` tells you plainly when it is not: "Connector integration
  web-search is not available for this account."
- A region that offers the connector: `us-east-1`, `eu-west-1` or `ap-northeast-1`. A
  region outside that set reports the same "not available for this account" message,
  so check the region before concluding the account is not enabled.
- Model access in Amazon Bedrock for the model this notebook uses.
- An AgentCore Gateway with a web search target on it. The next section creates one,
  and the toolkit itself never creates infrastructure.

## Citations are a condition of use

The AgentCore web search terms require that source links are kept and shown in
anything an end user sees. The agent below is prompted to cite URLs for that reason,
and the tool hands the model a title, a URL and a publication date alongside each
extract so it can.

## Setup

In [ ]:
%pip install -q langgraph langchain "langchain-aws[tools]"

In [2]:
import json
import os
from datetime import datetime, timedelta, timezone

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

from langchain_aws.tools import create_web_search_toolkit

#: The three regions that offer the web search connector.
WEB_SEARCH_REGIONS = ("us-east-1", "eu-west-1", "ap-northeast-1")

# Set deliberately rather than read from AWS_REGION. An ambient region of, say,
# us-west-2 creates a gateway happily and then fails the target with "connector
# integration web-search is not available for this account", which reads as an
# entitlement problem when the real cause is the region.
REGION = "us-east-1"
if REGION not in WEB_SEARCH_REGIONS:
    raise ValueError(f"web search is offered in {', '.join(WEB_SEARCH_REGIONS)}")


def answer_text(message):
    """Return the text of a message whose content may be a list of blocks.

    A reasoning model answers with a list that also carries reasoning blocks, so
    printing `message.content` directly would show those as well.
    """
    if isinstance(message.content, str):
        return message.content
    return "\n".join(
        block["text"]
        for block in message.content
        if isinstance(block, dict) and block.get("type") == "text"
    )


# Cross region inference profile. Any Bedrock model you have access to works.
MODEL_ID = "bedrock_converse:us.anthropic.claude-sonnet-5"

## The gateway and the web search target

Web search reaches the service as a Gateway connector target, so a gateway has to
exist before the toolkit can be pointed at one. Two IAM permissions are involved, and
they sit on different principals:

- The **gateway's own service role** needs `bedrock-agentcore:InvokeWebSearch` on
  `arn:aws:bedrock-agentcore:<region>:aws:tool/web-search.v1`, which is the
  service owned connector, plus `bedrock-agentcore:InvokeGateway` on
  `arn:aws:bedrock-agentcore:<region>:<account>:gateway/*`. Its trust policy allows
  `bedrock-agentcore.amazonaws.com` to assume it.
- **Your own credentials**, the ones the toolkit signs with, need
  `bedrock-agentcore:InvokeGateway` on the gateway you are calling.

Set `GATEWAY_ROLE_ARN` to a role with that trust policy and those permissions. The
cell below is idempotent: it reuses the gateway and the target if they already exist,
so re-running the notebook does not pile up resources.

In [3]:
from bedrock_agentcore.gateway.client import GatewayClient

GATEWAY_NAME = "langchain-aws-web-search-sample"
TARGET_NAME = "amazon-web-search"  # the SDK's default target name

gateway_client = GatewayClient(region_name=REGION)

gateway = gateway_client.get_gateway_by_name(GATEWAY_NAME)
if gateway is None:
    gateway = gateway_client.create_gateway_and_wait(
        name=GATEWAY_NAME,
        roleArn=os.environ["GATEWAY_ROLE_ARN"],
        authorizerType="NONE",  # inbound auth is SigV4
        protocolType="MCP",
    )
GATEWAY_ID = gateway["gatewayId"]

if gateway_client.get_gateway_target_by_name(GATEWAY_ID, TARGET_NAME) is None:
    target = gateway_client.create_web_search_target(gateway_identifier=GATEWAY_ID)
    print(f"created target {target['name']}: {target['status']}")

print(f"gateway {GATEWAY_ID} is ready with a {TARGET_NAME} target")

gateway langchain-aws-web-search-sample-ggjyfhepar is ready with a amazon-web-search target


## Create the toolkit

Unlike the browser and code interpreter toolkits in this package, this factory is
synchronous, because a search does not need a session to be started first.

Passing `target_name` is optional. Gateway prefixes every tool with the name of the
target it came from, so supplying the name lets the client compute the tool name
instead of spending a `tools/list` round trip discovering it.

In [4]:
toolkit, tools = create_web_search_toolkit(
    region=REGION,
    gateway_id=GATEWAY_ID,
    target_name=TARGET_NAME,
)

search_tool = tools[0]
print(f"tool: {search_tool.name}")
print(f"arguments: {list(search_tool.args_schema.model_fields)}")

tool: web_search
arguments: ['query', 'max_results', 'include_domains', 'exclude_domains', 'published_after', 'published_before']


## A first search, without an agent

The tool is callable on its own, which is the quickest way to see the shape of what
the model will be reading. Each result is a numbered block with a title, a URL, a
publication date when the index reports one, and an extract.

In [5]:
print(
    search_tool.invoke(
        {
            "query": "Amazon Bedrock AgentCore web search connector regions",
            "max_results": 2,
        }
    )
)

1. Domain and publish date filters for Web Search on AgentCore
   URL: https://aws.amazon.com/blogs/machine-learning/domain-and-publish-date-filters-for-web-search-on-agentcore/
   Published: 05:00PM, Tuesday, August 18 2026, PDT
   Today, we’re announcing runtime domain and published-date filtering for Web Search on Amazon Bedrock AgentCore, a platform to build, connect, and optimize agents at scale with any framework or model. This capability ships as part of the web-search connector version 1.2.0. These capabilities give developers per-call control over which web domains their agents can search and what publication-date window results must fall within, all enforced server-side. No external orchestration is required. When combined with existing admin-level domain policies, organizations have a layered filtering model that enforces enterprise governance while giving individual API calls the flexibility to narrow scope dynamically, per request. 

Alongside runtime filtering, this relea

## Build the agent

The system prompt does two jobs: it tells the model to search rather than answer from
memory when a question is time sensitive, and it makes citing the URLs part of the
task. The second one is not a stylistic preference, it is the acceptable use condition
from the top of this notebook.

In [6]:
SYSTEM_PROMPT = """You are a research assistant with access to web search.

Search the web whenever a question touches recent events, releases, prices or
anything else that may have changed since your training data. Do not answer from
memory in those cases.

Every fact you take from a search result must carry the source URL next to it. End
your answer with the list of URLs you used. If the results do not answer the
question, say so instead of filling the gap."""

model = init_chat_model(MODEL_ID)
agent = create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [7]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "When did Web Search on Amazon Bedrock AgentCore become "
                    "generally available, and which regions offer it?"
                ),
            }
        ]
    }
)

print(answer_text(result["messages"][-1]))

## Web Search on Amazon Bedrock AgentCore — GA Date and Regions

**General Availability:** Web Search on Amazon Bedrock AgentCore became generally available on **June 17, 2026** ([AWS News Blog](https://aws.amazon.com/blogs/aws/announcing-web-search-on-amazon-bedrock-agentcore-ground-your-ai-agents-in-current-accurate-web-knowledge/)). At launch it was available in only one region: **US East (N. Virginia) — us-east-1** ([AWS What's New](https://aws.amazon.com/about-aws/whats-new/2026/06/amazon-bedrock-agentcore-web-search/)).

**Regional expansion:** On **August 18–19, 2026**, AWS expanded Web Search availability with a new connector version (1.2.0) that added runtime domain and published-date filtering, and extended the service to two additional regions:
- **Europe (Ireland) — eu-west-1**
- **Asia Pacific (Tokyo) — ap-northeast-1**

This brought total coverage to three regions: US East (N. Virginia), Europe (Ireland), and Asia Pacific (Tokyo) ([AWS What's New](https://aws.amazon.com/a

## What the agent actually asked for

The interesting part of a search agent is the query it chose and the filters it set,
so read the tool call and not only the answer.

In [8]:
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        print(f"tool call -> {call['name']}({json.dumps(call['args'])})")
    if message.type == "tool":
        first_block = message.content.split("\n\n")[0]
        print(f"first result the model saw:\n{first_block}\n")

tool call -> web_search({"query": "Web Search Amazon Bedrock AgentCore generally available regions"})
first result the model saw:
1. Announcing Web Search on Amazon Bedrock AgentCore: Ground your AI agents in current, accurate web knowledge
   URL: https://aws.amazon.com/blogs/aws/announcing-web-search-on-amazon-bedrock-agentcore-ground-your-ai-agents-in-current-accurate-web-knowledge/
   Published: 08:00AM, Wednesday, June 17 2026, PDT
   Channy Yun (윤석찬)17 JUN 2026Amazon BedrockAmazon Bedrock AgentCoreAmazon Machine LearningArtificial IntelligenceAWS Summit New York     Add AWS as a preferred source on search Today, we’re announcing the general availability of Web Search on Amazon Bedrock AgentCore, a fully managed tool that enables agents to ground responses in current, cited web knowledge with zero data egress from customer’s secured AWS environment. 



## Narrowing to sources you trust

`include_domains` restricts a search to the domains you name, and a root domain also
matches its subdomains. It can only narrow a search, never widen it. This matters for
questions where a plausible looking blog post is worse than no answer, and it is the
argument to reach for when an agent keeps citing aggregators.

The model can set the filter itself, and the tool description tells it to prefer
official documentation, but pinning the domain in the prompt makes the run
reproducible.

In [9]:
docs_only = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Using only docs.aws.amazon.com as the source domain, what "
                    "credential provider does the AgentCore web search connector "
                    "target accept?"
                ),
            }
        ]
    }
)

print(answer_text(docs_only["messages"][-1]))

Based on AWS documentation at docs.aws.amazon.com, the AgentCore Web Search connector target does not require you to configure or manage any outbound credentials at all.

**Key finding:** The Web Search Tool connector is a **built‑in, AWS‑owned connector**. According to the AgentCore devguide page for the connector, "There are no search APIs to provision, no outbound credentials to manage, and no result‑parsing glue to maintain," and "the Gateway authenticates to the AWS‑owned connector and routes the request internally" — the customer never supplies API keys or OAuth credentials for this target (docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html).

This matches the general AgentCore Gateway authorization model for **Connector-type targets** (which includes Web Search, since it is added via `connectorId: "web-search"`): for Lambda, API Gateway, Smithy, and **Connector targets**, the only credential provider type used is:

```
{ "credenti

## Only recent pages

`published_after` and `published_before` take ISO-8601 UTC timestamps and filter on
the publication date the index reports. Pages with no date are dropped by the filter,
so a narrow window can come back with fewer results than expected.

In [10]:
since = (datetime.now(timezone.utc) - timedelta(days=60)).strftime("%Y-%m-%dT%H:%M:%SZ")

print(
    search_tool.invoke(
        {
            "query": "Amazon Bedrock AgentCore announcements",
            "max_results": 3,
            "published_after": since,
        }
    )
)

1. ICYMI: What landed for AI builders in August 2026
   URL: https://aws.amazon.com/blogs/machine-learning/icymi-what-landed-for-ai-builders-in-august-2026/
   Published: 01:01PM, Wednesday, September 09 2026, PDT
   A recap of the biggest Amazon Bedrock, AgentCore, and Strands updates from August 2026. At AWS, we have long focused on making foundational technologies accessible and providing the infrastructure needed to put them to work. Amazon Bedrock, used by more than 225,000 active customers, including over 80% of Fortune 100 companies, embodies that commitment by providing access to leading models and a broad set of tools to build and scale AI, with the security, reliability, performance, and cost efficiency customers need in production. As agentic applications expand, Amazon Bedrock extends this foundation with AgentCore, which lets you build, connect, and optimize agents using any framework and model. AWS also released the Strands Agent Harness SDK as open source, giving you the

## Watching the steps as they happen

Streaming in `updates` mode shows each search as the agent runs it, which is what you
want behind a UI: the user sees that a search is happening rather than waiting on a
blank screen.

In [11]:
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the maximum query length the AgentCore web search tool accepts?",
            }
        ]
    },
    stream_mode="updates",
):
    for node, update in chunk.items():
        for message in update.get("messages", []):
            calls = getattr(message, "tool_calls", []) or []
            for call in calls:
                print(f"[{node}] searching: {call['args']['query']}")
            if message.type == "tool":
                print(f"[{node}] got {message.content.count('   URL: ')} results")
            elif not calls and answer_text(message):
                print(f"[{node}] answer:\n{answer_text(message)}")

[model] searching: AgentCore web search tool maximum query length characters


[tools] got 10 results


[model] answer:
According to the AWS documentation for Amazon Bedrock AgentCore's Web Search Tool, the `query` field must be **200 characters or fewer** ("Must be 200 characters or fewer") when invoked through the `tools/call` MCP interface.

Source: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html


## Clean up

`toolkit.close()` releases the HTTP connection the client holds. It does not touch the
gateway, which is yours and is meant to outlive any one agent.

Set `DELETE_GATEWAY = True` to remove the gateway and the target this notebook
created. The IAM role is left alone either way, since it was not created here.

In [12]:
toolkit.close()

DELETE_GATEWAY = False

if DELETE_GATEWAY:
    target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, TARGET_NAME)
    if target:
        gateway_client.delete_gateway_target_and_wait(
            gatewayIdentifier=GATEWAY_ID, targetId=target["targetId"]
        )
    gateway_client.delete_gateway_and_wait(gatewayIdentifier=GATEWAY_ID)
    print(f"deleted {GATEWAY_ID}")
else:
    print("toolkit closed, gateway left in place")

toolkit closed, gateway left in place
